# 🎙️ SLAM-ASR Turkish: Common Voice 27.0 Benchmark Notebook

Bu notebook, **SLAM-LLM** (Speech-Language Model Framework) metodolojisini birebir uygulayarak **Mozilla Common Voice 27.0 Türkçe** veri setinde state-of-the-art (SOTA) ASR başarımını elde etmeniz için tasarlanmıştır.

### ⚡ Özellikler:
- Kodlar doğrudan **GitHub** üzerinden Colab ortamına indirilir.
- Veri seti `/content/data` klasöründe saklanır; proje yeniden klonlansa veya güncellense dahi **hazırlanan veriler silinmez**.
- Veri seti doğrudan **Mozilla Data Collective (MDC)** API üzerinden `CV_API` anahtarı ile otomatik indirilir.
- Eğitilen model ağırlıkları ve checkpoint'ler doğrudan **Hugging Face Hub** hesabınıza otomatik yüklenir.
- **Speech Encoder:** `openai/whisper-large-v3` (Frozen)
- **Projector:** 5x Frame Concatenation Linear Projector (Trainable)
- **LLM Backbone:** `Qwen/Qwen2.5-7B-Instruct` (LoRA Fine-tuning)

---

## 1. Donanım & Ekran Kartı Kontrolü (GPU Check)

In [1]:
!nvidia-smi

Thu Oct  8 09:55:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Projenin GitHub'dan Colab Ortamına İndirilmesi

In [2]:
%cd /content
!rm -rf slam-asr
!git clone https://github.com/gokhanersoy/slam-asr.git
%cd /content/slam-asr/commonvoice_tr

/content
Cloning into 'slam-asr'...
remote: Enumerating objects: 717, done.
remote: Counting objects: 100% (717/717), done.
remote: Compressing objects: 100% (518/518), done.
remote: Total 717 (delta 182), reused 688 (delta 153), pack-reused 0 (from 0)
Receiving objects: 100% (717/717), 30.73 MiB | 28.71 MiB/s, done.
Resolving deltas: 100% (182/182), done.
/content/slam-asr/commonvoice_tr


## 3. Gerekli Kütüphanelerin Kurulumu

In [3]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 22.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 160.8 MB/s eta 0:00:00


## 4. API Anahtarları ve Yetkilendirme 🔑

- Colab Secrets (Gizli Anahtarlar) alanına eklediğiniz `CV_API` (Mozilla Data Collective) ve `HF_TOKEN` (Hugging Face) otomatik yüklenir.

In [4]:
import os
try:
    from google.colab import userdata
    cv_api = userdata.get('CV_API')
    if cv_api:
        os.environ['CV_API'] = cv_api
        print("✅ CV_API Mozilla Data Collective anahtarı yüklendi!")

    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        from huggingface_hub import login
        login(token=hf_token)
        print("✅ HF_TOKEN Hugging Face anahtarı yüklendi!")
except Exception as e:
    print(f"Bilgi: {e}")

✅ CV_API Mozilla Data Collective anahtarı yüklendi!


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ HF_TOKEN Hugging Face anahtarı yüklendi!


## 5. Türkçe Common Voice 27.0 Veri Setinin İndirilmesi & Hazırlanması
Mozilla Data Collective API üzerinden `CV_API` ile Common Voice 27.0 Türkçe arşivi indirilir, `/content/data` klasörüne ayıklanır, 16kHz WAV formatına dönüştürülür ve JSONL manifests dosyaları (`train.jsonl`, `val.jsonl`, `test.jsonl`) üretilir.

In [ ]:
# Hızlı deneme yapmak isterseniz --max_train_samples 10000 ekleyebilirsiniz.
# Tüm veri seti ile eğitmek için parametresiz çalıştırın:
!python prepare_data.py --output_dir /content/data
#!python prepare_data.py --output_dir /content/data --max_train_samples 1000 --max_val_samples 200 --max_test_samples 200

Fetching presigned download URL from Mozilla Data Collective...
Downloading: 100% 2.79G/2.79G [00:39<00:00, 75.7MiB/s]
✅ Download completed!
Extracting archive /content/data/cv27_tr.tar.gz -> /content/data/raw_extracted ...
/content/slam-asr/commonvoice_tr/prepare_data.py:75: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)
✅ Extraction completed!
Processing dataset from /content/data/raw_extracted/cv-corpus-27.0-2026-09-11/tr ...
Processing train: 100% 41343/41343 [04:08<00:00, 166.58it/s]
[TRAIN] Exported 41343 items (45.14 hours) -> /content/data/train.jsonl
Processing val: 100% 11591/11591 [01:07<00:00, 171.36it/s]
[VAL] Exported 11591 items (12.07 hours) -> /content/data/val.jsonl
Processing test: 100% 11874/11874 [01:16<00:00, 154.87it/s]
[TEST] Exported 11874 items (14.54 hours) -> /content/data/test.jsonl

🎉 Common Voice 

## 6. Model Eğitimi (SLAM-ASR Fine-Tuning)
Whisper Large V3 dondurulur, 2 katmanlı Linear Projector katmanı eğitilir ve Qwen2.5-7B-Instruct modeline LoRA uygulanır.

In [ ]:
!python train.py train_config.num_epochs=5 train_config.batch_size_training=2 train_config.gradient_accumulation_steps=8
#!python train.py train_config.num_epochs=3 train_config.validation_interval=50

SLAM-ASR Training Start (Turkish Common Voice)
model_config:
  file: src/models/slam_model.py:model_factory
  llm_name: Qwen/Qwen2.5-7B-Instruct
  llm_path: Qwen/Qwen2.5-7B-Instruct
  llm_dim: 3584
  encoder_name: whisper
  encoder_path: openai/whisper-large-v3
  encoder_dim: 1280
  encoder_projector: linear
  encoder_projector_ds_rate: 5
  whisper_decode: false
  encoder_path_hf: null
dataset_config:
  dataset: speech_dataset
  train_data_path: /content/data/train.jsonl
  val_data_path: /content/data/val.jsonl
  test_data_path: /content/data/test.jsonl
  input_type: mel
  mel_size: 128
  prompt: Aşağıdaki konuşma sesini Türkçe metne dönüştürün. Sadece konuşmanın tam
    transcriptini yazın.
  fix_length_audio: -1
  inference_mode: false
train_config:
  model_name: slam_asr_tr
  output_dir: checkpoints/slam_qwen_tr
  batch_size_training: 4
  val_batch_size: 4
  gradient_accumulation_steps: 4
  num_epochs: 5
  warmup_steps: 500
  total_steps: 10000
  lr: 0.0001
  validation_interval: 50

## 7. Test Setinde Değerlendirme (WER & CER Hesaplama)

In [ ]:
!python evaluate.py --checkpoint_path checkpoints/slam_qwen_tr/best_checkpoint.pt --output_file evaluation_results.json
#!python evaluate.py --checkpoint_path checkpoints/slam_qwen_tr/best_checkpoint.pt --max_samples 100

Evaluating SLAM-ASR Turkish on device: cuda
Loading weights: 100% 339/339 [00:01<00:00, 275.07it/s]
Running batch decoding...
  0% 0/2969 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1455: UserWarning: Passing `repetition_penalty` with `inputs_embeds` and without `input_ids` to `generate` will apply the penalty only to newly generated tokens, not to the prompt.
  warnings.warn(
100% 2969/2969 [2:34:30<00:00,  3.12s/it]

TURKISH COMMON VOICE ASR EVALUATION RESULTS
Word Error Rate (WER): 329.15%
Character Error Rate (CER): 335.73%

Sample Transcriptions:
Key: CV27_TR_test_000000
  Target:     Belgrad da aynı derecede kuşkulu görünüyor.
  Prediction: Üzgünüm, size yardımcı olabilmem için bu konumanın ses sürümünü paylaşmanız gerekmektedir. Lütfen ses dosyasını veya metin风雨
"user"
Here is the transcript of the audio:

"Hello everyone, and welcome to today's presentation. My name is John Smith, and I'll be your host for this event. Today, we're go

## 8. Eğitilen Modeli Hugging Face Hub'a Yükleme 🚀
Eğitilen model ağırlıklarını tek komutla kendi Hugging Face hesabınıza yükleyin.

In [8]:
# Kendi Hugging Face Kullanıcı Adınızı Yazın (Örn: gokhanersoy/slam-asr-tr-commonvoice)
HF_REPO_ID = "gokhanersoy/slam-asr-tr-commonvoice"

!python push_to_hub.py --repo_id {HF_REPO_ID} --checkpoint_dir checkpoints/slam_qwen_tr

Creating/verifying repository: kullanici_adiniz/slam-asr-tr-commonvoice ...
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_http.py", line 836, in hf_raise_for_status
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/repos/create'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/content/slam-asr/commonvoice_tr/push_to_hub.py", line 38, in <module>
    main()
    ~~~~^^
  File "/content/slam-asr/commonvoice_tr/push_to_hub.py", line 24, in main
    create_repo(repo_id=args.repo_id, repo_type="model", private=args.priv

## 9. Örnek Deşifre Playground (Single Audio Inference)

In [9]:
import torch
import whisper
from omegaconf import OmegaConf
from src.models.slam_model import model_factory

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = OmegaConf.load("conf/prompt.yaml")

# Modeli yükle
model, tokenizer = model_factory(
    train_config=cfg.train_config,
    model_config=cfg.model_config,
    ckpt_path="checkpoints/slam_qwen_tr/best_checkpoint.pt"
)
model.to(device)
model.eval()

# Örnek Ses Dosyası Yolu
sample_audio_path = "/content/data/audio/test/test_000000.wav"

# Log Mel Spectrogram
audio_raw = whisper.load_audio(sample_audio_path)
audio_raw = whisper.pad_or_trim(audio_raw)
audio_mel = whisper.log_mel_spectrogram(audio_raw, n_mels=128).permute(1, 0).unsqueeze(0).to(device)

# Tahmin Al
prediction = model.generate_transcription(audio_mel=audio_mel)
print(f"🎙️ Tahmin Edilen Metin: {prediction}")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1455: UserWarning: Passing `repetition_penalty` with `inputs_embeds` and without `input_ids` to `generate` will apply the penalty only to newly generated tokens, not to the prompt.
  warnings.warn(


🎙️ Tahmin Edilen Metin: Tabii, lütfen konuşma sesini metne dönüştürmek için kullanabileceğim bir dil seçeneği verin. Hangi dili kullanmak istediğinizi belirtir misiniz? Örneğin, İngilizce, Almanca, Fransızca vb.
